# Breast Cancer Subtype Classification
### EfficientNet-B0 Fine-tuning Pipeline

This notebook trains an EfficientNet-B0 model to classify 8 breast tumor subtypes:
- **Benign:** Adenosis (A), Fibroadenoma (F), Phyllodes Tumor (PT), Tubular Adenoma (TA)
- **Malignant:** Ductal Carcinoma (DC), Lobular Carcinoma (LC), Mucinous Carcinoma (MC), Papillary Carcinoma (PC)

---
## 1. Variables and Setup

In [ ]:
import copy
import time
import torch
import os
import re
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
# Fetch variables from the global scope (useful if running in a notebook environment) 
# or fall back to default values.
data_dir = globals().get("data_dir", "breast")
batch_size = globals().get("batch_size", 32)
num_epochs = 100
lr = globals().get("lr", 1e-3)
val_ratio = globals().get("val_ratio", 0.2)
num_workers = globals().get("num_workers", 0)# Crashes when i set it anything else.

# Store the original ImageFolder reference 
ORIGINAL_IMAGEFOLDER = datasets.ImageFolder

---
## 2. Custom Dataset: `BreastSubtypeDataset`

Custom Dataset class designed to extract specific breast tumor subtypes from messy filenames or directory names using regex and an alias dictionary.

In [ ]:
class BreastSubtypeDataset(Dataset):
    """
    Custom Dataset class designed to extract specific breast tumor subtypes 
    from messy filenames or directory names using regex and an alias dictionary.
    Inheriting from torch.utils.data.Dataset requires implementing __init__, __len__, and __getitem__.
    """
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self._fallback_dataset = None

        # Define the 8 target breast cancer subtypes we want to classify
        self.classes = ["A", "F", "PT", "TA", "DC", "LC", "MC", "PC"]
        # Create a mapping from class string to an integer index (required for PyTorch loss functions)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        self.targets = []

        # Map common filename strings/variations to our 8 standardized target classes
        aliases = {
            "ADENOSIS": "A", "A": "A",
            "FIBROADENOMA": "F", "F": "F",
            "PHYLLODES": "PT", "PT": "PT",
            "TUBULARADENOMA": "TA", "TUBULAR": "TA", "TA": "TA",
            "DUCTALCARCINOMA": "DC", "DUCTAL": "DC", "DC": "DC",
            "LOBULARCARCINOMA": "LC", "LOBULAR": "LC", "LC": "LC",
            "MUCINOUSCARCINOMA": "MC", "MUCINOUS": "MC", "MC": "MC",
            "PAPILLARYCARCINOMA": "PC", "PAPILLARY": "PC", "PC": "PC",
        }

        def infer_tumor_type(path, fname):
            """Attempts to deduce the tumor type from the file path or name."""
            stem = os.path.splitext(fname)[0]
            # Split filename and directory name by underscores, hyphens, or spaces to isolate keywords
            tokens = [t.upper() for t in re.split(r"[_\-\s]+", stem) if t]
            dir_tokens = [t.upper() for t in re.split(r"[_\-\s]+", os.path.basename(os.path.dirname(path))) if t]
            all_tokens = tokens + dir_tokens

            # Check if any token directly matches a class or an alias
            for t in all_tokens:
                if t in self.class_to_idx: return t
            for t in all_tokens:
                if t in aliases: return aliases[t]

            # Fallback: check specific positional parts of the filename (e.g., expecting "ID_Date_Subtype_...")
            parts = fname.split("_")
            if len(parts) >= 3:
                t = parts[2].upper()
                if t in self.class_to_idx: return t
                if t in aliases: return aliases[t]

            return None

        # Recursively search the directory for valid images
        valid_ext = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
        for dirpath, _, filenames in os.walk(root):
            for fname in filenames:
                # Ignore files that do not have recognized image extensions
                if not fname.lower().endswith(valid_ext):
                    continue
                path = os.path.join(dirpath, fname)
                tumor_type = infer_tumor_type(path, fname)
                
                # Skip images where the subtype cannot be inferred from the filename/folder
                if tumor_type is None:
                    continue

                # Store the file path alongside its integer label
                target = self.class_to_idx[tumor_type]
                self.samples.append((path, target))
                self.targets.append(target)

        # PyTorch ImageFolder conventions look for 'self.imgs'
        self.imgs = self.samples 

        # If custom parsing fails entirely (list is empty), fallback to standard PyTorch ImageFolder logic
        # (which assumes folder names are exactly the class names)
        if len(self.samples) == 0:
            folder_ds = ORIGINAL_IMAGEFOLDER(root=root, transform=transform)
            if len(folder_ds) == 0:
                raise RuntimeError(f"No valid images found in '{root}'.")
            self._fallback_dataset = folder_ds
            self.samples = folder_ds.samples
            self.targets = [y for _, y in folder_ds.samples]
            self.classes = folder_ds.classes
            self.class_to_idx = folder_ds.class_to_idx
            self.imgs = self.samples
            print("Warning: subtype labels not found in filenames; using folder labels via ImageFolder.")

    def __len__(self):
        # Returns the total number of samples; required by PyTorch DataLoader to know when an epoch ends
        if self._fallback_dataset is not None:
            return len(self._fallback_dataset)
        return len(self.samples)

    def __getitem__(self, index):
        # Fetches a single sample and label at the specified index; called dynamically during training
        if self._fallback_dataset is not None:
            return self._fallback_dataset[index]

        path, target = self.samples[index]
        # Open image and ensure it has 3 channels (RGB) regardless of original format (e.g., grayscale)
        img = Image.open(path).convert("RGB")
        # Apply any specified PyTorch transforms (augmentations/tensors)
        if self.transform is not None:
            img = self.transform(img)
        return img, target

datasets.ImageFolder = BreastSubtypeDataset

---
## 3. Hardware & Pretrained Weights Setup

In [ ]:
# Setup hardware accelerator: Use Nvidia GPU if available, otherwise fall back to CPU processing
device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
weights = globals().get("weights", EfficientNet_B0_Weights.DEFAULT)

preset = weights.transforms()
mean = getattr(preset, "mean", (0.485, 0.456, 0.406))
std = getattr(preset, "std", (0.229, 0.224, 0.225))

---
## 4. Data Transforms & Loaders

In [ ]:
# Training transforms include heavy augmentation to prevent overfitting
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # Randomly zoom and crop to 224x224 (EfficientNet default)
    transforms.RandomHorizontalFlip(),                   # 50% chance to flip left-right
    transforms.RandomVerticalFlip(),                     # 50% chance to flip up-down
    transforms.RandomRotation(15),                       # Rotate image between -15 and 15 degrees
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05), # Randomly tweak colors slightly
    transforms.ToTensor(),                               # Convert PIL Image (0-255) to PyTorch Tensor (0.0-1.0)
    transforms.Normalize(mean=mean, std=std),            # Apply the extracted ImageNet normalization constants
])

# Validation transforms only resize and normalize (no random augmentations)
# We need to evaluate the model on standard, undistorted images
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

In [ ]:
# Load the full dataset
full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
num_classes = len(full_dataset.classes)

# Determine the number of images that will go into training vs. validation sets
val_size = int(len(full_dataset) * val_ratio)
train_size = len(full_dataset) - val_size

# Split the dataset randomly into mutually exclusive train and validation subsets
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Deepcopy ensures that modifying the validation dataset's transforms 
# doesn't accidentally affect the training dataset. We then overwrite the val transforms.
val_dataset.dataset = copy.deepcopy(full_dataset)
val_dataset.dataset.transform = val_transform

# Create DataLoaders for efficient batching and multi-processing
# DataLoader handles shuffling, batching (e.g., 32 images at a time), and sending data to memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
# We don't need to shuffle validation data since order doesn't affect metric calculation
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Device: {device}")
print(f"Total samples: {len(full_dataset)}")
print(f"Classes ({len(full_dataset.classes)}): {full_dataset.classes}")

---
## 5. Class Weight Calculation for Imbalance

Dataset is imbalanced. We calculate inverse frequency weights so the loss function penalizes errors on rare classes more heavily.

In [ ]:
cnt = Counter(full_dataset.targets)
print("Class counts:", {full_dataset.classes[k]: v for k, v in sorted(cnt.items())})

total_samples = sum(cnt.values())
# Formula: Weight = Total / (Num_Classes * Count_of_Class) 
# Frequent classes get a weight < 1, rare classes get a weight > 1
class_weights = [total_samples / (num_classes * cnt[i]) for i in range(num_classes)]
# Move the weights tensor to the GPU/CPU so the loss function can access it
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

---
## 6. Model Setup & Unfreezing Top Layers

In [ ]:
# Initialize the EfficientNet model and load the pre-trained weights
model = models.efficientnet_b0(weights=weights)

# Freeze the entire network initially to preserve pre-trained feature extractors
# setting requires_grad = False freezes the layers
for p in model.parameters():
    p.requires_grad = False

# Unfreeze the later stages of the network. This allows the model to learn 
# dataset-specific high-level features while keeping low-level edge/texture detectors frozen.
# model.features[5:] accesses the deeper residual blocks of the EfficientNet architecture.
for p in model.features[5:].parameters():
    p.requires_grad = True

# Replace the final classification head to match our 8 target classes instead of ImageNet's 1000
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

---
## 7. Optimization & Learning Rate Scheduler

In [ ]:
# Pass the class weights to the loss function to handle the dataset imbalance
# CrossEntropyLoss combines nn.LogSoftmax() and nn.NLLLoss() in one single class
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = optim.Adam([
    {'params': model.features[5:].parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': lr}
])

# Reduces the learning rate by half if validation loss stops improving for 3 epochs
# This helps the model "settle" into a local minimum when it plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

best_acc = 0.0
best_state = copy.deepcopy(model.state_dict())
start = time.time()

---
## 8. Training Loop & Early Stopping

In [ ]:
patience = 15
epochs_no_improve = 0

# Dictionaries to store metrics for plotting later
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': []
}

for epoch in range(num_epochs):
    # Set model to training mode (enables dynamic behaviors like dropout and batchnorm updates)
    model.train() 
    train_loss, train_correct, train_total = 0.0, 0, 0

    # Iterate over batches of images and labels from the training DataLoader
    for bi, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device) # Move data to the same device as the model
        
        optimizer.zero_grad()      # Clear old gradients from the previous step so they don't accumulate
        logits = model(x)          # Forward pass: pass inputs through the network to get raw output scores
        loss = criterion(logits, y)# Compute loss: compare raw scores to actual true labels
        loss.backward()            # Backward pass: compute gradients of the loss w.r.t model parameters
        optimizer.step()           # Update weights: adjust parameters based on calculated gradients

        # Accumulate metrics: loss.item() gets the scalar value, multiply by batch size to un-average it
        train_loss += loss.item() * x.size(0)
        # argmax(1) finds the index of the highest logit score (the predicted class)
        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += y.size(0)

        # Print early feedback during the first epoch
        if epoch == 0 and bi % 20 == 0:
            print(f"[epoch 1] batch {bi}/{len(train_loader)} loss={loss.item():.4f}")

    # Set model to evaluation mode (disables dropout, freezes batchnorm statistics for consistent testing)
    model.eval() 
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    # torch.no_grad() disables gradient calculation to save memory and compute time during inference
    with torch.no_grad(): 
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            # Accumulate validation metrics
            val_loss += loss.item() * x.size(0)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += y.size(0)

    # Calculate average metrics for the epoch by dividing total accumulated values by dataset size
    tr_loss = train_loss / train_total
    tr_acc = train_correct / train_total
    va_loss = val_loss / val_total
    va_acc = val_correct / val_total
    
    # Save metrics to our history dictionary for plotting
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    scheduler.step(va_loss)

    # Check for improvement and save the best model weights into memory
    if va_acc > best_acc:
        best_acc = va_acc
        # deepcopy ensures we save a distinct copy of the weights, not just a reference
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        # Increment counter if validation accuracy didn't improve
        epochs_no_improve += 1

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
        f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}"
    )

    # Halt training if the model hasn't improved in 'patience' epochs to save time and prevent overfitting
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered! Validation accuracy hasn't improved for {patience} epochs.")
        break

---
## 9. Save Best Model Checkpoint

In [ ]:
# Load the best weights back into the model before saving it to disk
model.load_state_dict(best_state)
# Save a dictionary (checkpoint) containing the weights, class names, and performance
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "classes": full_dataset.classes,
        "best_val_acc": best_acc,
    },
    "efficientnet_b0_breast_best.pth",
)

print(f"Training done in {(time.time() - start)/60:.1f} min. Best val acc: {best_acc:.4f}")

---
## 10. Visualization: Loss & Accuracy Curves

Plot Loss and Accuracy Curves to visually check for overfitting (train and val diverging) or underfitting.

In [ ]:
plt.figure(figsize=(14, 5))

# Subplot 1: Loss curves
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Subplot 2: Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

---
## 11. Visualization: Confusion Matrix

Generate Confusion Matrix to see which specific classes are being misclassified as others.

In [ ]:
print("Generating Confusion Matrix on Validation Set...")
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        logits = model(x)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y.numpy())

cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=full_dataset.classes, 
            yticklabels=full_dataset.classes)
plt.title('Confusion Matrix (Best Model)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()